# 0) Imports & Repo Root (Local Fine-Tuning)
This notebook fine-tunes a **local gpt-oss model on your own GPU** using the yt_factlink dataset.

We reuse the same data splits and prompts as the OpenAI fine-tune notebook, but all training happens locally (no OpenAI API calls).

In [ ]:
import json, time
from pathlib import Path

# Find repo root
def find_repo_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else Path(start).resolve()
    for p in [start] + list(start.parents):
        if (p / ".git").exists() or (p / "README.md").exists():
            return p
    raise RuntimeError(f"Repo root not found from start={start}")

repo_root = find_repo_root()
print("Repo root:", repo_root)

# 不再使用 OpenAI API key；之後的訓練完全在本地 GPU 上進行。

# 1) Paths & Split & Model Selection
Here you choose the split and provider/model settings.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import re

# ------------------------------------------------------------------
# Dataset root
# ------------------------------------------------------------------
dataset_root = repo_root / "datasets" / "yt_factlink"
print("Dataset root:", dataset_root)

# ------------------------------------------------------------------
# Choose split
# ------------------------------------------------------------------
SPLIT_NAME = "split_v1_seed42"   # <--- Change here to switch splits

# ------------------------------------------------------------------
# Choose provider + base model (LOCAL GPT-OSS)
# ------------------------------------------------------------------
PROVIDER    = "local_gpt_oss"
MODEL_NAME  = "openai/gpt-oss-20b"   # local base model id on HF Hub
SUFFIX_TAG  = "all_at_once"          # your experiment name

# This suffix will be used in run folder naming (not sent to OpenAI)
FINETUNE_SUFFIX = f"{SUFFIX_TAG}_{SPLIT_NAME}"

# ------------------------------------------------------------------
# Split paths
# ------------------------------------------------------------------
SPLIT_DIR   = dataset_root / "01_conversion" / "03_splits" / SPLIT_NAME
TRAIN_JSONL = SPLIT_DIR / "train.data.jsonl"
TEST_JSONL  = SPLIT_DIR / "test.data.jsonl"

assert SPLIT_DIR.exists(),   f"Missing split folder: {SPLIT_DIR}"
assert TRAIN_JSONL.exists(), f"Missing train split: {TRAIN_JSONL}"
assert TEST_JSONL.exists(),  f"Missing test split: {TEST_JSONL}"

print("Using split:", SPLIT_NAME)
print("Train split:", TRAIN_JSONL)
print("Test  split:", TEST_JSONL)

# ------------------------------------------------------------------
# Prompt paths
# ------------------------------------------------------------------
PROMPT_DIR  = dataset_root / "02_prompts" / SUFFIX_TAG
SYSTEM_PATH = PROMPT_DIR / "system.txt"
USER_PATH   = PROMPT_DIR / "user.txt"

assert SYSTEM_PATH.exists(), f"Missing system prompt: {SYSTEM_PATH}"
assert USER_PATH.exists(),   f"Missing user prompt: {USER_PATH}"

print("Prompts loaded from:", PROMPT_DIR)

# ------------------------------------------------------------------
# Create global "runs" folder next to this notebook
# Then create ONE folder for this specific run inside it.
# ------------------------------------------------------------------

import re

def slugify(s: str) -> str:
    s = re.sub(r"[^a-zA-Z0-9._-]+", "-", s)
    s = re.sub(r"-{2,}", "-", s)
    return s.strip("-")

# Folder where the notebook lives
NOTEBOOK_DIR = Path.cwd()

# Create folder for this model (next to this notebook)
MODEL_DIR = NOTEBOOK_DIR / slugify(MODEL_NAME)
MODEL_DIR.mkdir(exist_ok=True)

# Add .gitkeep at the model folder level
(MODEL_DIR / ".gitkeep").write_text("")

# Create runs/ folder inside model folder
RUNS_ROOT = MODEL_DIR / "runs"
RUNS_ROOT.mkdir(exist_ok=True)

# Add .gitkeep inside runs folder
(RUNS_ROOT / ".gitkeep").write_text("")

# Timestamp-based unique run name
timestamp = datetime.now(timezone.utc).strftime("%Y-%m-%d_%H%M%S")

# Smart run folder name
run_name = "_".join([
    timestamp,
    slugify(PROVIDER),
    slugify(MODEL_NAME),
    slugify(SUFFIX_TAG),
    slugify(SPLIT_NAME),
])

# 5) Create the run folder inside runs/
RUN_DIR = RUNS_ROOT / run_name
RUN_DIR.mkdir(parents=True, exist_ok=False)


print("Notebook directory:", NOTEBOOK_DIR.resolve())
print("Global runs folder:", RUNS_ROOT.resolve())
print("Run directory     :", RUN_DIR.resolve())

# ------------------------------------------------------------------
# Derived training file — stored INSIDE the run folder
# ------------------------------------------------------------------
OPENAI_TRAIN_FILE = RUN_DIR / "train.openai.jsonl"
print("Derived training file will be:", OPENAI_TRAIN_FILE)

# Summary
print("\n=== Configuration OK ===")
print("Base model         :", MODEL_NAME)
print("Provider           :", PROVIDER)
print("Finetune Suffix    :", FINETUNE_SUFFIX)
print("Run directory      :", RUN_DIR)


# 2) Load Split + Prompts
Load train split and the frozen finetuning prompt pair.

In [ ]:
def read_jsonl(path: Path):
    with path.open("r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

train_rows = read_jsonl(TRAIN_JSONL)
print("Train examples:", len(train_rows))

system_template = SYSTEM_PATH.read_text(encoding="utf-8")
user_template   = USER_PATH.read_text(encoding="utf-8")

# ---------------------------------------------------------------
# Copy prompt files into RUN_DIR/01_inputs for reproducibility
# ---------------------------------------------------------------
INPUTS_DIR = RUN_DIR / "01_inputs"
INPUTS_DIR.mkdir(parents=True, exist_ok=True)

# Write frozen copies
(INPUTS_DIR / "system.txt").write_text(system_template, encoding="utf-8")
(INPUTS_DIR / "user.txt").write_text(user_template, encoding="utf-8")

print("Copied prompts to:", INPUTS_DIR.resolve())

# 3) Build OpenAI Fine-Tuning JSONL
Convert each canonical example into messages format.

In [ ]:
# Add .gitkeep inside individual run folder
(RUN_DIR / ".gitkeep").write_text("")

# Create subfolder for this run's inputs
INPUTS_DIR = RUN_DIR / "01_inputs"
INPUTS_DIR.mkdir(parents=True, exist_ok=True)

# Path for the training file used for fine-tuning
OPENAI_TRAIN_FILE = INPUTS_DIR / "train.openai.jsonl"

print("Input directory created at:", INPUTS_DIR.resolve())
print("Training file will be:", OPENAI_TRAIN_FILE.resolve())

# ---------------------------------------------------------
# Build the OpenAI training JSONL
# ---------------------------------------------------------

def build_user_message(r):
    msg = user_template
    msg = msg.replace("[VIDEOTITLE]",   r.get("VIDEOTITEL", ""))
    msg = msg.replace("[VIDEOCONTEXT]", r.get("VIDEOCONTEXT", ""))
    msg = msg.replace("[TARGETCOMMENT]", r.get("TARGETCOMMENT", ""))
    return msg

openai_lines = []
for r in train_rows:
    openai_lines.append({
        "messages": [
            {"role": "system",    "content": system_template},
            {"role": "user",      "content": build_user_message(r)},
            {"role": "assistant", "content": json.dumps(r["labels"], ensure_ascii=False)}
        ]
    })

# Write the file
with OPENAI_TRAIN_FILE.open("w", encoding="utf-8") as f:
    for obj in openai_lines:
        f.write(json.dumps(obj, ensure_ascii=False) + "\n")

print("\nWrote training JSONL to:", OPENAI_TRAIN_FILE)
print("Example entry:", openai_lines[0])


# 4) Build Local HF Dataset for QLoRA
Convert the canonical train split + prompts into a text-only dataset for local fine-tuning (no JSONL upload).

In [ ]:
from datasets import Dataset, DatasetDict
import random

# 將每一筆 canonical example 轉成單一 text 欄位（類似 tmp.ipynb 的格式）
def format_example(r):
    msg = user_template
    msg = msg.replace("[VIDEOTITLE]",   r.get("VIDEOTITEL", ""))
    msg = msg.replace("[VIDEOCONTEXT]", r.get("VIDEOCONTEXT", ""))
    msg = msg.replace("[TARGETCOMMENT]", r.get("TARGETCOMMENT", ""))

    return (
        "[SYSTEM]\n" + system_template + "\n[/SYSTEM]\n"
        + "[USER]\n" + msg + "\n[/USER]\n"
        + "[ASSISTANT]\n" + json.dumps(r["labels"], ensure_ascii=False)
    )

texts = [format_example(r) for r in train_rows]
print("Total formatted examples:", len(texts))

# 簡單 train/validation split
indices = list(range(len(texts)))
random.seed(42)
random.shuffle(indices)
val_ratio = 0.05
val_size = max(1, int(len(texts) * val_ratio))
val_idx = set(indices[:val_size])

train_texts = [texts[i] for i in range(len(texts)) if i not in val_idx]
val_texts   = [texts[i] for i in range(len(texts)) if i in val_idx]

local_ds = DatasetDict({
    "train": Dataset.from_dict({"text": train_texts}),
    "validation": Dataset.from_dict({"text": val_texts}),
})

print(local_ds)
LOCAL_DATA_DIR = RUN_DIR / "01_inputs" / "yt_factlink_train_data"
local_ds.save_to_disk(str(LOCAL_DATA_DIR))
print("Saved local dataset to:", LOCAL_DATA_DIR)

# 5) Load gpt-oss-20b & Apply LoRA (QLoRA style)
Load the base `openai/gpt-oss-20b` model and wrap it with LoRA adapters for lightweight fine-tuning.

In [ ]:
!pip install peft

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, Mxfp4Config
from peft import LoraConfig, get_peft_model
from datasets import load_from_disk

# ==================================================================
# 檢查 GPU 可用性（AMD MI300x / ROCm）
# ==================================================================
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(f"GPU device count: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")
        print(f"    Memory: {torch.cuda.get_device_properties(i).total_memory / 1024**3:.2f} GB")
else:
    print("⚠️  WARNING: No GPU detected! Training will be very slow on CPU.")
    print("   For AMD MI300x, ensure ROCm is installed and PyTorch is built with ROCm support.")

BASE_MODEL_ID = MODEL_NAME  # "openai/gpt-oss-20b"
TOKENIZER_ID = BASE_MODEL_ID

LOCAL_DATA_DIR = RUN_DIR / "01_inputs" / "yt_factlink_train_data"
print("\nLoading dataset from:", LOCAL_DATA_DIR)
dataset = load_from_disk(str(LOCAL_DATA_DIR))

print("\nLoading tokenizer & base model:", BASE_MODEL_ID)

tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_ID, use_fast=True)

quant_config = Mxfp4Config(dequantize=True)
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    trust_remote_code=True,
    quantization_config=quant_config,
    dtype=torch.bfloat16,
    device_map="auto",
    low_cpu_mem_usage=True,
)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)

# 確保模型在訓練模式
model.train()

# 檢查模型設備分配
print("\n" + "="*60)
print("Model device allocation:")
if hasattr(model, "hf_device_map"):
    for name, device in model.hf_device_map.items():
        print(f"  {name[:50]:<50} -> {device}")
else:
    # 手動檢查參數所在設備
    devices = set()
    for name, param in list(model.named_parameters())[:5]:
        if param.is_cuda:
            devices.add(f"cuda:{param.device.index}")
        else:
            devices.add("cpu")
    print(f"  Parameters on: {', '.join(devices)}")

# 檢查可訓練參數
print("\nModel with LoRA adapters ready.")
print("Model training mode:", model.training)
print("Any trainable param:", any(p.requires_grad for p in model.parameters()))

# 顯示前幾個參數的 requires_grad 狀態（debug 用）
print("\nSample parameter requires_grad status:")
for name, p in list(model.named_parameters())[:10]:
    print(f"{name[:60]:<60} {p.requires_grad}")

# 6) Train Locally with SFTTrainer
Run supervised fine-tuning (SFT) on the local dataset using TRL's `SFTTrainer`.

In [ ]:
!pip install trl

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer

OUTPUT_DIR = RUN_DIR / "02_outputs" / "gpt-oss-20b-lora"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 確保模型在訓練模式
model.train()
print("Model training mode:", model.training)

# 再次確認有可訓練參數
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {trainable_params:,}")

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=2,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=16,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    logging_steps=20,
    eval_steps=200,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=2,
    bf16=True,
    gradient_checkpointing=False,  # 先關閉，避免與量化模型衝突
    report_to="none",
)


def formatting_func(example):
    return example["text"]


trainer = SFTTrainer(
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    args=training_args,
    formatting_func=formatting_func,
    processing_class=tokenizer,
)

train_output = trainer.train()
train_output

# 7) Save Local Fine-Tuned Model & Artifacts
Save the LoRA adapter, tokenizer, and basic training metadata into the run folder.

In [ ]:
import shutil

# ------------------------------------------------------------------
# Create outputs and snapshots directories
# ------------------------------------------------------------------
OUTPUTS_DIR = RUN_DIR / "02_outputs"
SNAPSHOTS_DIR = RUN_DIR / "03_snapshots"

OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
SNAPSHOTS_DIR.mkdir(parents=True, exist_ok=True)

print("Outputs dir:   ", OUTPUTS_DIR.resolve())
print("Snapshots dir: ", SNAPSHOTS_DIR.resolve())

# ------------------------------------------------------------------
# Save fine-tune results into outputs/
# ------------------------------------------------------------------
# 儲存 LoRA adapter + tokenizer
trainer.model.save_pretrained(str(OUTPUT_DIR))
tokenizer.save_pretrained(str(OUTPUT_DIR))

# 儲存簡單的訓練摘要
summary = {
    "base_model": BASE_MODEL_ID,
    "output_dir": str(OUTPUT_DIR),
    "num_train_epochs": training_args.num_train_epochs,
    "learning_rate": training_args.learning_rate,
    "train_samples": len(dataset["train"]),
    "val_samples": len(dataset["validation"]),
    "train_metrics": getattr(train_output, "metrics", {}),
}

(OUTPUTS_DIR / "local_finetune_summary.json").write_text(
    json.dumps(summary, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

# ------------------------------------------------------------------
# Snapshot the notebook itself into 03_snapshots
# ------------------------------------------------------------------
NOTEBOOK_PATH = Path("run_finetune.ipynb")

if NOTEBOOK_PATH.exists():
    shutil.copy2(NOTEBOOK_PATH, SNAPSHOTS_DIR / "run_finetune_snapshot.ipynb")
    print("Notebook snapshot saved to:", (SNAPSHOTS_DIR / "run_finetune_snapshot.ipynb").resolve())
else:
    print("WARNING: run_finetune.ipynb not found in current directory. Cannot snapshot notebook.")

print("\nSaved artifacts into:", OUTPUTS_DIR.resolve())
print("Full run folder     :", RUN_DIR.resolve())
